# Retrofit Document Types (Non-Bank Only)

Updates ground truth `DOCUMENT_TYPE` with model predictions **only for non-bank-statement images**.
Bank statement rows are never modified.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# --- Configure paths ---
GROUND_TRUTH_CSV = Path("../evaluation_data/bank/ground_truth_bank.csv")
RESULTS_DIR = Path("../evaluation_data/output/batch")

# Auto-find latest results CSV (match both naming conventions)
results_files = sorted(
    f for f in RESULTS_DIR.glob("batch_*_results*.csv") if "summary" not in f.name
)
if not results_files:
    # Fallback: try legacy naming
    results_files = sorted(RESULTS_DIR.glob("batch_results_*.csv"))
assert results_files, f"No batch results CSV found in {RESULTS_DIR}"
RESULTS_CSV = results_files[-1]
print(f"Ground truth: {GROUND_TRUTH_CSV}")
print(f"Results CSV:  {RESULTS_CSV}")

In [ ]:
# Load both CSVs
gt = pd.read_csv(GROUND_TRUTH_CSV, dtype=str)
results = pd.read_csv(RESULTS_CSV, dtype=str)

print(f"Ground truth columns: {list(gt.columns)}")
print(f"Results columns:      {list(results.columns)}")

# Find image column in each CSV
IMAGE_COL_CANDIDATES = ["image_name", "image_file", "filename", "file"]
gt_image_col = next(c for c in IMAGE_COL_CANDIDATES if c in gt.columns)
results_image_col = next(c for c in IMAGE_COL_CANDIDATES if c in results.columns)
print(f"\nGT image column:      {gt_image_col}")
print(f"Results image column: {results_image_col}")
print(f"Ground truth: {len(gt)} rows")
print(f"Results:      {len(results)} rows")

# Show existing DOCUMENT_TYPE distribution
if "DOCUMENT_TYPE" in gt.columns:
    print("\nExisting ground truth DOCUMENT_TYPE:")
    print(gt["DOCUMENT_TYPE"].value_counts(dropna=False))
else:
    print("\nNo DOCUMENT_TYPE column in ground truth yet")

In [ ]:
# Build predicted type map keyed by basename (strips any path prefixes)
type_map = {
    Path(k).name: v
    for k, v in results.set_index(results_image_col)["document_type"].to_dict().items()
}

# Update only non-bank rows, tracking each remapping
updated = 0
skipped_bank = 0
no_prediction = 0
remappings = []  # (image, old_type, new_type)

if "DOCUMENT_TYPE" not in gt.columns:
    gt["DOCUMENT_TYPE"] = pd.NA

for idx, row in gt.iterrows():
    image_name = Path(row[gt_image_col]).name  # normalize to basename
    existing_type = str(row.get("DOCUMENT_TYPE", "")).strip().upper()
    predicted_type = type_map.get(image_name)

    # Never touch bank statements
    if existing_type == "BANK_STATEMENT":
        skipped_bank += 1
        continue

    if predicted_type:
        old_type = row.get("DOCUMENT_TYPE", "")
        gt.loc[idx, "DOCUMENT_TYPE"] = predicted_type
        remappings.append((image_name, old_type, predicted_type))
        updated += 1
    else:
        no_prediction += 1

print(f"Updated:        {updated}")
print(f"Skipped (bank): {skipped_bank}")
print(f"No prediction:  {no_prediction}")

# Show all remappings for review
if remappings:
    print(f"\n--- Remappings ({len(remappings)}) ---")
    remap_df = pd.DataFrame(remappings, columns=["image", "old_type", "new_type"])
    remap_df["changed"] = remap_df["old_type"] != remap_df["new_type"]
    display(remap_df)

In [ ]:
# Final distribution
print("Updated DOCUMENT_TYPE distribution:")
print(gt["DOCUMENT_TYPE"].value_counts(dropna=False))
print()
gt[[gt_image_col, "DOCUMENT_TYPE"]]

In [ ]:
# Save — overwrites original ground truth
gt.to_csv(GROUND_TRUTH_CSV, index=False)
print(f"Saved: {GROUND_TRUTH_CSV}")